# Modello MOIRAI (Zero-Shot Foundation Model) - Bonds & Macro data

Testiamo **MOIRAI** (Masked Encoder-based Universal Time Series Representation Learning) sui dati combinati Macro-Bond. MOIRAI è un *Foundation Model* sviluppato da Salesforce per le serie storiche, che effettua inferenza *zero-shot*.

In [4]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

# Importiamo Moirai e GluonTS
from gluonts.dataset.pandas import PandasDataset
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

print("Librerie per MOIRAI caricate correttamente.")


/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.6.1, so it will not be used.
  warnings.warn(


Librerie per MOIRAI caricate correttamente.


Prepariamo il dataset. Poiché abbiamo molti bond (molti `isincode`), dovremo preparare un dataset GluonTS indicando `isincode` come identificativo (item_id) in un `PandasDataset`.

In [ ]:
# Carichiamo i dati
df_moirai = pd.read_csv('./data/df_bond_macro.csv', index_col=0).reset_index()

if 'index' in df_moirai.columns:
    df_moirai = df_moirai.rename(columns={'index': 'isincode'})

df_moirai['referencedate'] = pd.to_datetime(df_moirai['referencedate'])
df_moirai = df_moirai.sort_values(['referencedate', 'isincode'])

FORECAST_HORIZON = 30
target_col = 'Yield_to_Maturity'

print("Inizializzazione del modello MOIRAI...")
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-base"),
    prediction_length=FORECAST_HORIZON,
    context_length=300,
    patch_size="auto",
    num_samples=100,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)
print("Modello MOIRAI inizializzato.")


Inizializzazione del modello MOIRAI...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/365M [00:00<?, ?B/s]

Modello MOIRAI inizializzato.


Eseguiamo l'inferenza. Tratteremo alcuni ISIN di test nei periodi di test stabiliti dalla split time-series. Data la pesantezza computazionale di valutare centinaia di bond con un foundation model su CPU, campioneremo un sottoinsieme di ISIN e punti nel tempo.

In [ ]:
unique_dates = np.sort(df_moirai['referencedate'].unique())
n_splits = 5
test_size_days = len(unique_dates) // (n_splits + 2)
gap_days = FORECAST_HORIZON

moirai_roc_auc_scores = []
moirai_f1_scores = []

# Campioniamo solo alcuni ISIN per ridurre i tempi di inferenza in questo notebook di esempio
np.random.seed(42)
all_isins = df_moirai['isincode'].unique()
sample_isins = np.random.choice(all_isins, min(40, len(all_isins)), replace=False)
print(f"Valutiamo MOIRAI su un sample di {len(sample_isins)} ISIN per velocizzare il processo.")

df_sample = df_moirai[df_moirai['isincode'].isin(sample_isins)].copy()
df_sample = df_sample.set_index('referencedate')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

predictor = model.create_predictor(batch_size=10, device=device)

for fold in range(n_splits):
    print(f"\n--- Fold {fold+1} ---")

    test_start_idx = len(unique_dates) - test_size_days * (n_splits - fold) - (FORECAST_HORIZON - 1)
    test_end_idx = test_start_idx + test_size_days
    test_dates = unique_dates[test_start_idx:test_end_idx]

    valid_test_dates = test_dates[:-FORECAST_HORIZON]
    if len(valid_test_dates) < 10:
        valid_test_dates = test_dates

    cutoff_dates = valid_test_dates[::max(1, len(valid_test_dates)//10)][:10]

    y_preds_prob = []
    y_trues = []

    for cutoff_date in cutoff_dates:
        future_idx = np.where(unique_dates == cutoff_date)[0][0] + FORECAST_HORIZON
        if future_idx >= len(unique_dates):
            continue
        future_date = unique_dates[future_idx]

        for isin in sample_isins:
            df_isin = df_sample[df_sample['isincode'] == isin]

            history_df = df_isin[df_isin.index <= cutoff_date]
            if len(history_df) < 50:
                continue

            future_data = df_isin[df_isin.index >= future_date]
            if future_data.empty:
                continue

            current_val = history_df.iloc[-1][target_col]
            future_val = future_data.iloc[0][target_col]

            true_target = 1 if future_val < current_val else 0

            hist_ts = history_df[[target_col]].copy()
            hist_ts = hist_ts[~hist_ts.index.duplicated(keep='last')]
            hist_ts = hist_ts.asfreq('B').ffill()

            ds = PandasDataset(hist_ts, target=target_col)

            try:
                forecast_it = predictor.predict(ds)
                forecast = next(forecast_it)
                pred_horizon = forecast.samples[:, -1]
                prob_up = np.mean(pred_horizon > current_val)

                y_preds_prob.append(prob_up)
                y_trues.append(true_target)

            except Exception as e:
                print(f" [Debug] Errore MOIRAI su {isin} al {cutoff_date.date()}: {str(e)[:100]}")

    if len(y_trues) == 0:
        print("Nessuna previsione valida in questo fold.")
        continue

    print(f"  -> Trovate {len(y_trues)} previsioni valide per questo fold.")

    # Controllo di sicurezza
    if len(np.unique(y_trues)) > 1:
        roc = roc_auc_score(y_trues, y_preds_prob)
    else:
        roc = np.nan
        print("  [Avviso] Solo una classe presente in questo fold. ROC-AUC = NaN")

    # SOGLIA FISSA A 0.5
    y_preds_class = [1 if p > 0.5 else 0 for p in y_preds_prob]
    f1 = f1_score(y_trues, y_preds_class)

    if not np.isnan(roc):
        moirai_roc_auc_scores.append(roc)

    moirai_f1_scores.append(f1)

    print(f"Fold ROC-AUC: {roc:.3f} | F1: {f1:.3f} ")

print(f"\n=======================================================")
print(f"Risultato Finale MOIRAI (Media su {len(moirai_roc_auc_scores)} folds):")
if moirai_roc_auc_scores:
    print(f"Mean ROC-AUC: {np.mean(moirai_roc_auc_scores):.3f}")
    print(f"Mean F1-Score: {np.mean(moirai_f1_scores):.3f}")
else:
    print("Nessun punteggio calcolato.")
print(f"=======================================================")

Valutiamo MOIRAI su un sample di 40 ISIN per velocizzare il processo.
Using device: cuda

--- Fold 1 ---
  -> Trovate 300 previsioni valide per questo fold.
Fold ROC-AUC: 0.670 | F1: 0.701 

--- Fold 2 ---
  -> Trovate 310 previsioni valide per questo fold.
Fold ROC-AUC: 0.487 | F1: 0.722 

--- Fold 3 ---
  -> Trovate 324 previsioni valide per questo fold.
Fold ROC-AUC: 0.459 | F1: 0.493 

--- Fold 4 ---
  -> Trovate 346 previsioni valide per questo fold.
Fold ROC-AUC: 0.511 | F1: 0.378 

--- Fold 5 ---
  -> Trovate 352 previsioni valide per questo fold.
Fold ROC-AUC: 0.364 | F1: 0.359 

Risultato Finale MOIRAI (Media su 5 folds):
Mean ROC-AUC: 0.498
Mean F1-Score: 0.531


## Modello MOIRAI (Zero-Shot Foundation Model) - MACRO DATA ONLY

In [7]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit

# Importiamo le librerie per Moirai e GluonTS
from gluonts.dataset.pandas import PandasDataset
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

print("Librerie per MOIRAI caricate correttamente.")


Librerie per MOIRAI caricate correttamente.


Prepariamo i dati. MOIRAI ha bisogno che il DataFrame abbia un DatetimeIndex con una frequenza specificata (es. 'B' per business days o 'D' per giorni). Useremo l'intero storico dell'Euribor 3M e faremo previsioni mobili per coprire gli stessi periodi di test dell'LSTM.

In [ ]:
# Carichiamo i dati originali
df_moirai = pd.read_csv('./data/df_macro_long.csv', index_col=0)
df_moirai.index = pd.to_datetime(df_moirai.index)
df_moirai = df_moirai.sort_index()

# Assicuriamo una frequenza (se ci sono buchi, GluonTS può lamentarsi, resampliamo a giorni lavorativi o riempiamo)
df_moirai = df_moirai.asfreq('B').ffill()

FORECAST_HORIZON = 90
target_col = 'Euribor_3M'

# Inizializziamo il modello MOIRAI
# Scegliamo la versione 'small' (14M params) per velocità di esecuzione su CPU
print("Inizializzazione del modello MOIRAI (scaricamento pesi se non presenti)...")
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-base"),
    prediction_length=FORECAST_HORIZON,
    context_length=500, # Quanta storia passata guardare
    patch_size="auto",
    num_samples=100, # Numero di sample per le probabilità (previsione probabilistica)
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)
print("Modello MOIRAI inizializzato.")


Inizializzazione del modello MOIRAI (scaricamento pesi se non presenti)...
Modello MOIRAI inizializzato.


Eseguiamo l'inferenza. Poiché Moirai è un modello molto pesante, valutarlo su ogni singolo giorno del test set richiederebbe troppo tempo. Effettueremo la valutazione campionando alcune date di test e usando il modello per prevedere i 90 giorni successivi.

In [9]:
# Creiamo una serie di dataset per GluonTS
# Per emulare lo split temporale in modo compatto, valutiamo MOIRAI sull'ultimo fold o su finestre specifiche

tscv = TimeSeriesSplit(n_splits=5, gap=FORECAST_HORIZON)

moirai_roc_aucs = []
moirai_f1_scores = []

X_base = df_moirai[[target_col]].values

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

predictor = model.create_predictor(batch_size=10, device=device)

for fold, (train_index, test_index) in enumerate(tscv.split(X_base)):
    print(f"\n--- FOLD {fold+1} ---")

    # Per non esplodere coi tempi di calcolo, prendiamo solo alcuni punti chiave nel test_index
    # Campioniamo 10 punti equispaziati nel test set da prevedere
    sample_indices = np.linspace(test_index[0], test_index[-1] - FORECAST_HORIZON, 10, dtype=int)

    y_preds_prob = []
    y_trues = []

    for idx in sample_indices:
        # La storia arriva fino a 'idx'
        history_df = df_moirai.iloc[:idx+1][[target_col]]

        # Il vero valore futuro a 30 giorni
        future_val = df_moirai.iloc[idx + FORECAST_HORIZON][target_col]
        current_val = history_df.iloc[-1][target_col]

        # Target direzionale reale
        true_target = 1 if future_val > current_val else 0
        y_trues.append(true_target)

        # Creiamo PandasDataset per GluonTS
        ds = PandasDataset(history_df, target=target_col)

        # Inferenza
        forecast_it = predictor.predict(ds)
        forecast = next(forecast_it)

        # forecast.samples ha shape (num_samples, prediction_length)
        # Prendiamo la previsione per il giorno 90 (indice 89)
        pred_90d = forecast.samples[:, -1]

        # Calcoliamo la probabilità che il valore al giorno 90 sia maggiore del valore attuale
        prob_up = np.mean(pred_90d > current_val)
        y_preds_prob.append(prob_up)

    # Calcolo Metriche
    roc = roc_auc_score(y_trues, y_preds_prob)


    y_preds_class = [1 if p > 0.5 else 0 for p in y_preds_prob]
    f1 = f1_score(y_trues, y_preds_class)

    print(f"Fold ROC-AUC: {roc:.3f} | F1: {f1:.3f}")

    moirai_roc_aucs.append(roc)
    moirai_f1_scores.append(f1)

print(f"\n=======================================================")
print(f"Risultato Finale MOIRAI (Media su {tscv.n_splits} folds):")
print(f"Mean ROC-AUC: {np.mean(moirai_roc_aucs):.3f}")
print(f"Mean F1-Score: {np.mean(moirai_f1_scores):.3f}")
print(f"=======================================================")


Using device: cuda

--- FOLD 1 ---
Fold ROC-AUC: 0.812 | F1: 0.875

--- FOLD 2 ---
Fold ROC-AUC: 0.604 | F1: 0.333

--- FOLD 3 ---
Fold ROC-AUC: 0.719 | F1: 0.500

--- FOLD 4 ---
Fold ROC-AUC: 0.250 | F1: 0.500

--- FOLD 5 ---
Fold ROC-AUC: 0.680 | F1: 0.444

Risultato Finale MOIRAI (Media su 5 folds):
Mean ROC-AUC: 0.613
Mean F1-Score: 0.531
